In [1]:
import pandas as pd
import numpy as np
rng = np.random.default_rng()
import matplotlib.pyplot as plt
import matplotlib as mpl
from skimage.measure import block_reduce
import sympy as sp
from sympy.stats import Normal, E , density, MultivariateNormal

#### Functions

In [2]:
def multivariate_lognormal_cascade(n, sigma1=1, sigma2=1, corr=0):
    mu1 = -1/2 * sigma1**2
    mu2 = -1/2 * sigma2**2

    PQ = np.exp(rng.multivariate_normal(np.array([mu1,mu2]),  np.array([[sigma1**2,corr*sigma1*sigma2], [corr*sigma1*sigma2, sigma2**2]]), 4))

    P = PQ[:,0].reshape(2,2)
    Q = PQ[:,1].reshape(2,2)
    for i in range(n):
        PQ = np.exp(rng.multivariate_normal(np.array([mu1,mu2]),np.array([[sigma1**2,corr*sigma1*sigma2], [corr*sigma1*sigma2, sigma2**2]]), P.shape[0]**2 * 4))
        P = np.kron(P, np.ones((2,2)))
        P = P * PQ[:,0].reshape(P.shape)
        Q = np.kron(Q, np.ones((2,2)))
        Q = Q * PQ[:,1].reshape(P.shape)

    P = P / np.sum(P)
    Q = Q / np.sum(Q)
    return(np.stack([P,Q], axis=-1))

def Theil(pop):
  tp = pop.sum(axis=2)
  prob = pop[:,:,:] / tp[:,:, np.newaxis]
  ep = np.sum(-np.log2(prob) * prob, axis=2)
  e = (ep * tp / 2).sum()
  return(1-e)

First, perform taylor serie of the entropy

In [3]:
Z, z = sp.symbols('Z z')

r1 = 1 / (1 + Z)
r2 = Z / (1 + Z)

H = - (r1 * sp.log(r1, 2) + r2 * sp.log(r2, 2)) / 2
H = H.subs({Z : sp.exp(z) })


# Développement d'ordre 5
DL = sp.series(H, z, 0, 7).removeO().expand()
DL

-z**6/(1152*log(2)) + z**4/(128*log(2)) - z**2/(16*log(2)) + 1/2

In [93]:
mu1, mu2 = sp.symbols('mu1 mu2')

m1 = (mu1 - mu2 + sigma1**2 - rho*sigma1*sigma2) * t
m2 = (mu1 - mu2 + rho*sigma1*sigma2 - sigma2**2) * t 

E1 = s2*t + m1**2
E2 = s2*t + m2**2

coeff2 = (E1 + E2)/2
coeff2 = coeff2.subs({mu1 : - sigma1**2/2, mu2 : - sigma2**2/2 })
coeff2 = sp.simplify(coeff2)
coeff2

t**2*(-2*rho*sigma1*sigma2 + sigma1**2 + sigma2**2)*(-2*rho*sigma1*sigma2 + sigma1**2 + sigma2**2 + 4)/4

In [82]:
mu1, mu2, sigma1, sigma2, rho = sp.symbols(
    'mu1 mu2 sigma1 sigma2 rho'
)

s2 = (sigma1**2 + sigma2**2 - 2*rho*sigma1*sigma2) * t

m1 = (mu1 - mu2 + sigma1**2 - rho*sigma1*sigma2) * t
m2 = (mu1 - mu2 + rho*sigma1*sigma2 - sigma2**2) * t 

E1 =  m1**4 + 6*m1**2*s2 + 3*s2**2
E2 = m2**4 + 6*m2**2*s2 + 3*s2**2

coeff4 = (E1 + E2) / 2 

coeff4 = result.subs({mu1 : - sigma1**2/2, mu2 : - sigma2**2/2 })

sp.collect(coeff4, t )

t**4*((-rho*sigma1*sigma2 + sigma1**2/2 + sigma2**2/2)**4/2 + (rho*sigma1*sigma2 - sigma1**2/2 - sigma2**2/2)**4/2) + t**3*(3*(-2*rho*sigma1*sigma2 + sigma1**2 + sigma2**2)*(-rho*sigma1*sigma2 + sigma1**2/2 + sigma2**2/2)**2 + 3*(-2*rho*sigma1*sigma2 + sigma1**2 + sigma2**2)*(rho*sigma1*sigma2 - sigma1**2/2 - sigma2**2/2)**2) + 3*t**2*(-2*rho*sigma1*sigma2 + sigma1**2 + sigma2**2)**2

## Autres approches

In [8]:
x1, x2 = sp.symbols('x_1 x_2')


H = - (sp.exp(x1) * sp.log(sp.exp(x1) / (sp.exp(x1) + sp.exp(x2)), 2) + sp.exp(x2) * sp.log(sp.exp(x2) / (sp.exp(x1) + sp.exp(x2)), 2)) / 2

n = 2
DL = 0

for i in range(n + 1):
    for j in range(n + 1 - i):
        deriv = sp.diff(H, x1, i, x2, j).subs({x1: 0, x2: 0})
        DL += deriv * x1**i * x2**j / (sp.factorial(i) * sp.factorial(j))

sp.expand(sp.simplify(DL))

-x_1**2/(8*log(2)) + x_1**2/4 + x_1*x_2/(4*log(2)) + x_1/2 - x_2**2/(8*log(2)) + x_2**2/4 + x_2/2 + 1

In [10]:
H

-exp(x_1)*log(exp(x_1)/(exp(x_1) + exp(x_2)))/(2*log(2)) - exp(x_2)*log(exp(x_2)/(exp(x_1) + exp(x_2)))/(2*log(2))

In [9]:
from sympy import symbols
from sympy.stats import Normal, E

mu, sigma = symbols('mu sigma', positive=True)

X = Normal('X', mu, sigma)

# Espérance
print(E(X))

# Deuxième moment
print(E(X**2))

# Troisième moment
print(E(X**3))

# Quatrième moment
print(E(X**4))

mu
mu**2 + sigma**2
mu**3 + 3*mu*sigma**2
mu**4 + 6*mu**2*sigma**2 + 3*sigma**4


In [ ]:
DL